# Introduction
## Project Overview

In this project we analyze data from ecommerce platform.  
The dataset contains millions of orders placed by thousands of customers.  

- User can place multiple orders
- Order contains multiple products  
- Products belong to aisles and departments

The objective of projetc is to tranform raw data into actionable business insights and predictive models that support smarter business decision.

## Business Questions

- Which products are reordered most frequently, and which are purchased the least? +

- Which aisles and departments contribute the largest number of orders? +

- When do customers usually place orders (day and hour)? +

- Which aisles are frequently purchased together? +

- After how many days without an order can we consider a customer at risk of churning? +

- Do customers fill their baskets more on weekends or during the week? +

- At how many items in a basket does a customer tend to stop adding products? +

- Is there any relationship between the last order and the reorder? +

- In how many days will this customer place their next order? (model)

- Which products are most often added to the shopping cart first? +

- Which products are most often added to the shopping cart together?

- Can we predict the reorder of a product by a specific buyer? +

- Can we predict a buyer's shopping cart based on their past purchases?

- Can we predict the number of items in a customer's next basket?

## Loading, editing, and merging datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import accuracy_score, roc_auc_score

aisles = pd.read_csv("datasets/aisles.csv")
departments = pd.read_csv("datasets/departments.csv")
order_products = pd.read_csv("datasets/order_products.csv")
orders = pd.read_csv("datasets/orders.csv")
products = pd.read_csv("datasets/products.csv")

print("orders:", orders.shape)
print("order_products:", order_products.shape)
print("products:", products.shape)
print("aisles:", aisles.shape)
print("departments:", departments.shape)


In [ ]:
orders["order_id"].nunique()

In [ ]:
order_products["order_id"].nunique()

In [ ]:
df = orders.merge(order_products, on="order_id", how="inner")
print("after first merge:", df.shape)

In [ ]:
df = df.merge(products, on="product_id", how="left")
print("after products:", df.shape)

In [ ]:
df=df.merge(aisles, on="aisle_id", how="left")
print("after aisles:", df.shape)

In [ ]:
df=df.merge(departments, on="department_id", how="left")
print("after department:", df.shape)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["days_since_prior_order"] = df["days_since_prior_order"].fillna(df["days_since_prior_order"].median())
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df = df.dropna(subset="product_id")
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df["order_hour_of_day"] = df["order_hour_of_day"].fillna(df["order_hour_of_day"].median())
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df = df.dropna()
df.isnull().sum().sort_values(ascending=False)
df.dtypes

In [ ]:
df.dtypes

In [ ]:
df.to_csv('datasets/cleaned_dataset.csv', index=False)

### Data Cleaning
- Missing 'days_since_prior_order' values were filled with the median.  
- Rows with missing product information were removed.  
- Missing 'order_hour_of_day' values were filled using the median hour.  

This ensures that the dataset remains consistent for analysis and modeling.

Which products are reordered most frequently, and which are purchased the least?

In [ ]:
# Aggregate product-level statistics
product_stats = (
    df.groupby(["product_id", "product_name"])
      .agg(
          total_purchases=("order_id", "count"),
          total_reorders=("reordered", "sum")
      )
      .reset_index()
)

# Compute reorder rate
product_stats["reorder_rate"] = (
    product_stats["total_reorders"] /
    product_stats["total_purchases"]
)

product_stats.head()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    product_stats["reorder_rate"],
    bins=40,
    kde=True
)

plt.title("Distribution of Product Reorder Rates")
plt.xlabel("Reorder Rate")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.show()

In [ ]:
least_purchased = (
    product_stats.sort_values("total_purchases")
                 .head(20)
)

plt.figure(figsize=(10,8))

sns.barplot(
    data=least_purchased,
    y="product_name",
    x="total_purchases"
)

plt.title("20 Least Purchased Products")
plt.xlabel("Total Purchases")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=product_stats,
    x="total_purchases",
    y="reorder_rate",
    alpha=0.3
)

plt.title("Sales Volume vs Reorder Rate")
plt.xlabel("Total Purchases")
plt.ylabel("Reorder Rate")
plt.tight_layout()
plt.show()

---
### Which aisles and departments contribute the largest number of orders?
In this section, we aim to identify which product aisles and departments generate the highest number of orders.  
To answer this question, we will use the merged dataset and group the data by aisle and department. Then, we will count the number of products ordered in each category.  
This analysis will help us understand which categories drive the majority of customer purchases and contribute the most to overall sales activity.

In [ ]:
df = pd.read_csv("datasets/cleaned_dataset.csv")
top_aisles = df.groupby("aisle")["order_id"].nunique().sort_values(ascending=False).head(15)
top_aisles

In [ ]:
top_departments = df.groupby("department")["order_id"].nunique().sort_values(ascending=False).head(15)
top_departments

In [ ]:
fig = px.bar(
    top_aisles.reset_index(),
    x="aisle",
    y="order_id",
    title="Top 10 Aisles by Number of Orders"
)

fig.show()

The bar chart shows the aisles contributing the largest number of orders.   

Fresh produce categories clearly dominate the ranking, with "fresh fruits" and "fresh vegetables" accounting for the highest number of orders. Dairy products such as yogurt and milk also appear among the most frequently purchased categories. 

This indicates that everyday grocery staples and fresh food products represent the core of customer purchasing behavior.  

In [ ]:
fig = px.bar(
    top_departments.reset_index(),
    x="department",
    y="order_id",
    title="Top 10 Departments by Number of Orders"
)


fig.show()

The bar chart shows which departments contribute the largest number of orders.  

The produce department clearly dominates the ranking, followed by dairy eggs and beverages. These categories represent essential grocery items that customers purchase regularly.  

This indicates that fresh food and everyday staples are the core drivers of customer purchases on the platform.

---

### When do customers usually place orders (day and hour)?


In [ ]:
orders_time = (df.groupby(["order_dow","order_hour_of_day"])["order_id"].nunique().reset_index(name="orders"))
pivot_orders = orders_time.pivot(index="order_dow", columns = "order_hour_of_day", values = "orders")

day_labels = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

pivot_orders.index = day_labels

fig = px.imshow (
    pivot_orders,
    color_continuous_scale = "YlOrRd",
    aspect ="auto",
    labels = dict(x="Hour of Day", y="Day of Week", color="Number of Orders"),
    title = "When do customers usually place orders?"
)

fig.update_layout(xaxis_title="Hour of Day", yaxis_title = "Day of Week (0=Sunday)")

fig.show()


The heatmap shows when customers most frequently place their orders during the week.  

Most orders are placed between approximately 8 AM and 4 PM, with a clear peak around late morning and early afternoon (10 AM – 2 PM).  

Sunday appears to be the most active day for grocery orders, followed by Monday. In contrast, very few orders are placed during nighttime hours.  

This pattern suggests that customers tend to shop during daytime hours, particularly at the beginning of the week, possibly when preparing for the upcoming week.  

---

### Which aisles are frequently purchased together?

In [ ]:
# Create aisle presence matrix per order
order_aisle_matrix = pd.crosstab(
    df["order_id"],
    df["aisle"]
)

# Convert counts to binary (presence / absence)
order_aisle_matrix = (order_aisle_matrix > 0).astype(int)

order_aisle_matrix.head()

In [ ]:
aisle_corr = order_aisle_matrix.corr()

aisle_corr.head()

In [ ]:
# Select top 20 most frequent aisles
top_aisles = (
    df["aisle"]
      .value_counts()
      .head(20)
      .index
)

corr_subset = aisle_corr.loc[top_aisles, top_aisles]

plt.figure(figsize=(12,10))

sns.heatmap(
    corr_subset,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Top 20 Aisles")
plt.tight_layout()
plt.show()

### After how many days without an order can we consider a customer at risk of churning?

In [ ]:
# Relationship between days since prior order and reorder probability

churn_analysis = (
    df.groupby("days_since_prior_order")
      .agg(
          avg_reorder_rate=("reordered", "mean"),
          number_of_orders=("order_id", "count")
      )
      .reset_index()
      .sort_values("days_since_prior_order")
)

churn_analysis.head()

In [ ]:
# Visualization
plt.figure(figsize=(10,6))

sns.lineplot(
    data=churn_analysis,
    x="days_since_prior_order",
    y="avg_reorder_rate",
    marker="o"
)

plt.title("Reorder Probability vs Days Since Previous Order")
plt.xlabel("Days Since Previous Order")
plt.ylabel("Average Reorder Probability")
plt.tight_layout()
plt.show()

### Do customers fill their baskets more on weekends or during the week?

In [ ]:
# Calculate basket size
basket_size = (
    df.groupby(["order_id", "order_dow"])
      .agg(basket_size=("add_to_cart_order", "max"))
      .reset_index()
)

basket_size.head()

In [ ]:
# Label weekends vs weekdays
basket_size["day_type"] = basket_size["order_dow"].apply(
    lambda x: "Weekend" if x in [0,6] else "Weekday"
)

basket_size.head()

In [ ]:
# Compute average basket size
basket_summary = (
    basket_size.groupby("day_type")
               .agg(
                   avg_basket_size=("basket_size", "mean"),
                   number_of_orders=("order_id", "count")
               )
               .reset_index()
)

basket_summary

In [ ]:
plt.figure(figsize=(8,6))

sns.barplot(
    data=basket_summary,
    x="day_type",
    y="avg_basket_size"
)

plt.title("Average Basket Size: Weekend vs Weekday")
plt.xlabel("Day Type")
plt.ylabel("Average Number of Items in Basket")
plt.tight_layout()
plt.show()


### At how many items in a basket does a customer tend to stop adding products?


In [ ]:
basket_size = df.groupby("order_id")["product_id"].count().reset_index()
basket_size.rename(columns={"product_id":"basket_size"}, inplace=True)
fig = px.histogram(
    basket_size,
    x="basket_size",
    nbins=40,
    title="Distribution of Basket Size",
    labels={"basket_size":"Number of Items in Basket"}
)

fig.show()

The histogram shows that most xuqtomers stop adding products when the basket reaches approximately 5-10 items.

---

### Is there a relationship between the last order and the reorder?

In [ ]:
reorder_by_days = df.groupby("days_since_prior_order")["reordered"].mean().reset_index()

fig = px.line(
    reorder_by_days,
    x="days_since_prior_order",
    y="reordered",
    title="Reorder Probability vs Days Since Last Order",
    labels={"days_since_prior_order":"Days Since Prior Order", "reordered":"Reoder Probability"}
)

fig.show()

The plot indicates a general negative relationship between the time since the previous order and the probability of reordering a product.  

Customers who place orders shortly after their previous purchase tend to have a higher reorder probability. As the number of days since the last order increases, the reorder probability gradually decreases.  

---

### After how many days without an order can we consider a customer at risk of churning?

In [ ]:
fig = px.histogram(
    df,
    x="days_since_prior_order",
    nbins=30,
    title = "Distribution of Days Between Orders",
    labels = {"days_sicne_proir_order":"Days Between Orders"}
)

fig.show()

To estimate when a customer may be at risk of churning, we analyze the distribution of the number of days between consecutive orders.  

The histogram shows that most customers place their next order within approximately 5 to 10 days after their previous purchase. The number of orders gradually decreases as the gap between orders increases.  

After around 20–25 days without placing a new order, the number of returning customers becomes significantly lower. This suggests that customers who have not ordered for more than about 3–4 weeks could be considered at risk of churning.  

There are also a large number of orders for the 30th and 31st days, based on which we can say that there are “monthly” orders.  

It is also important to note that the spike at 30 days occurs because the dataset caps the variable days_since_prior_order at 30 days.

---

### Is there any relationship between the last order and the reorder?

In [ ]:
first_products = df[df["add_to_cart_order"] == 1]

top_first_products = (
    first_products.groupby("product_name").size().reset_index(name="count").sort_values("count", ascending=False).head(20)
)

fig = px.bar(
    top_first_products,
    x = "count",
    y="product_name",
    orientation="h",
    title="Products Most Often Added to the Cart First",
    labels={
        "count":"Number of Times Added First",
        "product_name":"Product"
    },
    height=700
)

fig.update_layout(yaxis={"categoryorder":"total ascending"})

fig.show()

To understand how customers begin building their shopping baskets, we analyze the products that are most frequently added first to the cart.

The results show that staple grocery items such as bananas, milk, avocados, and berries are most often the first products added. Fresh produce clearly dominates the list.

This suggests that customers tend to start their shopping with commonly purchased essentials before adding other items.

From a business perspective, these products could be strategically highlighted on the homepage or placed prominently in stores to guide the rest of the shopping experience.

---

### Can we predict the reorder of a product by a specific buyer?

In [ ]:
def init_neural_ia(iteration = 500):
    """
    crée un model d'ia neural network et le renvoie
    """
    model = MLPClassifier(
        hidden_layer_sizes=(100),     # crée les "hiddden layers"
        activation='relu',                # fait en sorte que si les lien des neuronnes sont inferieur a 0 les mettent a 0
        solver='adam',                    # fait en sorte que plus le resultat est proche de la valeur attendue moins il fluctue les modificatio
        max_iter=iteration                # le nombre d'iteration
    )
    return model


def reorder_df(df):
    
    user_col = df.groupby('user_id').agg(
        user_total_orders = ('order_number', 'max'),
        user_moy_basket_size = ('product_id', 'count'),
        user_moy_reorder_days = ('days_since_prior_order', 'mean'),
    ).reset_index()

    # collones produit
    prod_col = df.groupby('product_id').agg(
        product_total_orders = ('order_id', 'count'),
        product_reorder_rate = ('reordered', 'mean'),
    ).reset_index()

    # Collones des 2
    fus_col = df.groupby(['user_id', 'product_id']).agg(
        fus_order_count = ('order_id', 'count'),
        fus_avg_cart_position = ('add_to_cart_order', 'mean'),
        fus_last_order = ('order_number', 'max'),
    ).reset_index()

    # Création de la nouvelle df
    df_new = (df[['user_id', 'product_id', 'reordered']]
                .merge(user_col, on='user_id', how='left')
                .merge(prod_col, on='product_id', how='left')
                .merge(fus_col, on=['user_id', 'product_id'], how='left')
                .drop_duplicates(subset=['user_id', 'product_id'])
                .reset_index(drop=True))
    return df_new

In [ ]:
df_reord = reorder_df(df)
df_reord

In [ ]:
inputt = df_reord.drop('reordered' , axis=1)
output = df_reord['reordered']

input_train, input_test, output_train, output_test = train_test_split(inputt, output, test_size=0.2, random_state=24, stratify=output)

In [ ]:
# 1. Configuration de la simulation
# On fait varier N (nombre d'échantillons) de 1 000 à 50 000
N = np.linspace(1, 7656670, 100)

# Constantes fixées (
d = 12            # Nombre de features (colonnes)
K = 100           # Nombre d'arbres (pour RFXGB et LGBM)
I = 20            # Nombre d'itérations (LogReg, NN)
W_nn = d * 100    # Poids du Neural Network


# Logistic regression= O(I*N*d)
logreg = I * N * d

# Random forest = O(K*N*log(N)*sqrt(d) )
rf = K * N * np.log2(N) * np.sqrt(d)

# xgboost = O(K*N*log(N)*d)
xgb = K * N * np.log2(N) * d

# Lightgbm= O(K * N_sample * d_bundle)
lgbm = K * (N * 0.2) * (d * 0.8)

# Neural network = O(I*N*W)
nn = I * N * W_nn


plt.figure(figsize=(12, 7))

# Tracer des courbes
plt.plot(N, lgbm, label='LightGBM')
plt.plot(N, logreg, label='Logistic regression')
plt.plot(N, rf, label='Random forest')
plt.plot(N, xgb, label='XGBoost')
plt.plot(N, nn, label='Neural network')

# Label
plt.title('Comparaison des ia')
plt.xlabel('Nombre d échantillons')
plt.ylabel('Opérations estimées')
plt.legend(fontsize=11)
plt.grid(True)

# Affichage
plt.show()

### Le meilleur modéle pour notre cas est le lightGBM

In [ ]:
def init_lightgbm(n_estimators=500, learning_rate=0.1):
    model = lgb.LGBMClassifier(
        n_estimators = n_estimators,
        learning_rate = learning_rate,
        verbose=-1
    )
    return model

In [ ]:
light_reorder = init_lightgbm()
light_reorder.fit(input_train, output_train)

In [ ]:
output_pred = light_reorder.predict(input_test)

print(f"Accuracy : {accuracy_score(output_test, output_pred):.2%}")

### Can we predict the number of items in a customer's next basket?

In [ ]:
#def basket_df(df):
   
#  order_col = (df.groupby('order_id').agg(
#     user_basket_size = ('product_id', 'count'),
#    reorder_rate = ("reordered", "mean"),
#).reset_index().sort_values(["user_id", "order_number"]))


In [ ]:
#output_pred = light_basket.predict(input_test)ad

#print(f"Accuracy : {accuracy_score(output_test, output_pred):.2%}")